# BGE Gemma Reranker Experiment

This notebook evaluates `BAAI/bge-reranker-v2-gemma` as an additional dense-top-100 reranker baseline on the BEA 2026 submission benchmark.

In [1]:
import os
import re
import unicodedata
from pathlib import Path
from typing import List, Tuple

import faiss
import numpy as np
import pandas as pd
import pyterrier as pt
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")

PROJECT_ROOT = Path("/Users/user/Submissions/BEA-2026").resolve()
QB_PATH = PROJECT_ROOT / "qbank.csv"
QUERIES_PATH = PROJECT_ROOT / "queries.csv"
QRELS_PATH = PROJECT_ROOT / "qrels.tsv"

ART_DIR = PROJECT_ROOT / "artifacts" / "bge_gemma_reranker"
ART_DIR.mkdir(parents=True, exist_ok=True)

DENSE_MODEL = "deutsche-telekom/gbert-large-paraphrase-cosine"
RERANKER_MODEL = "BAAI/bge-reranker-v2-gemma"
DENSE_TOPK = 100
FINAL_TOPK = 50
RERANK_BATCH = 4
MAX_LENGTH = 512

_ws_re = re.compile(r"\s+")
_tags_re = re.compile(r"<[^>]+>")


def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = _tags_re.sub(" ", s)
    s = unicodedata.normalize("NFKC", s)
    s = s.lower()
    s = re.sub(r"[^0-9a-zA-ZäöüßÄÖÜẞ]+", " ", s)
    s = _ws_re.sub(" ", s).strip()
    return s


def safe_concat(parts):
    clean = []
    for p in parts:
        if p is None:
            continue
        p = str(p)
        if p.strip() in ("", "N/A", "nan"):
            continue
        clean.append(p)
    return " ".join(clean)


def build_doc_text(row: pd.Series) -> str:
    return safe_concat([row.get("question"), row.get("choices_processed")])


def ensure_run(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "qid" not in df.columns or "docno" not in df.columns:
        raise ValueError("Run must contain columns: qid, docno")
    df["qid"] = df["qid"].astype(str)
    df["docno"] = df["docno"].astype(str)
    if "score" not in df.columns:
        df["score"] = 0.0
    df["score"] = df["score"].astype(float)
    if "rank" not in df.columns:
        df = df.sort_values(["qid", "score"], ascending=[True, False])
        df["rank"] = df.groupby("qid").cumcount() + 1
    return df


if not pt.java.started():
    pt.java.init()


class FaissDenseRetriever(pt.Transformer):
    def __init__(self, corpus_df: pd.DataFrame, model_name: str, topk: int, show_progress: bool = True):
        super().__init__()
        self.topk = int(topk)
        cdf = corpus_df[["docno", "text"]].copy()
        cdf["docno"] = cdf["docno"].astype(str)
        cdf["text"] = cdf["text"].astype(str)
        self.docnos = cdf["docno"].tolist()
        self.st = SentenceTransformer(model_name, device="cpu")
        xdoc = self.st.encode(
            cdf["text"].tolist(),
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=bool(show_progress),
        ).astype("float32")
        self.index = faiss.IndexFlatIP(xdoc.shape[1])
        self.index.add(xdoc)

    def transform(self, topics_df: pd.DataFrame) -> pd.DataFrame:
        qids = topics_df["qid"].astype(str).tolist()
        qs = topics_df["query"].astype(str).tolist()
        q = self.st.encode(qs, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
        scores, idxs = self.index.search(q, self.topk)
        rows = []
        for i, qid in enumerate(qids):
            for rank, (j, sc) in enumerate(zip(idxs[i], scores[i]), start=1):
                if j < 0:
                    continue
                rows.append({"qid": qid, "docno": self.docnos[j], "score": float(sc), "rank": int(rank)})
        return pd.DataFrame(rows)


def cut_k(k: int) -> pt.Transformer:
    return pt.apply.generic(lambda df: df.groupby("qid", sort=False).head(int(k)))


def add_query_col(topics_df: pd.DataFrame) -> pt.Transformer:
    tq = topics_df[["qid", "query"]].copy()
    tq["qid"] = tq["qid"].astype(str)

    def _merge(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["qid"] = df["qid"].astype(str)
        return df.merge(tq, on="qid", how="left")

    return pt.apply.generic(_merge)


def build_llm_inputs(pairs, tokenizer, prompt=None, max_length=512):
    if prompt is None:
        prompt = (
            "Given a query A and a passage B, determine whether the passage contains "
            "an answer to the query by providing a prediction of either 'Yes' or 'No'."
        )
    sep = "\n"
    prompt_inputs = tokenizer(prompt, return_tensors=None, add_special_tokens=False)["input_ids"]
    sep_inputs = tokenizer(sep, return_tensors=None, add_special_tokens=False)["input_ids"]
    inputs = []
    for query, passage in pairs:
        query_inputs = tokenizer(
            f"A: {query}",
            return_tensors=None,
            add_special_tokens=False,
            max_length=max_length * 3 // 4,
            truncation=True,
        )
        passage_inputs = tokenizer(
            f"B: {passage}",
            return_tensors=None,
            add_special_tokens=False,
            max_length=max_length,
            truncation=True,
        )
        item = tokenizer.prepare_for_model(
            [tokenizer.bos_token_id] + query_inputs["input_ids"],
            sep_inputs + passage_inputs["input_ids"],
            truncation="only_second",
            max_length=max_length,
            padding=False,
            return_attention_mask=False,
            return_token_type_ids=False,
            add_special_tokens=False,
        )
        item["input_ids"] = item["input_ids"] + sep_inputs + prompt_inputs
        item["attention_mask"] = [1] * len(item["input_ids"])
        inputs.append(item)
    return tokenizer.pad(
        inputs,
        padding=True,
        max_length=max_length + len(sep_inputs) + len(prompt_inputs),
        pad_to_multiple_of=8,
        return_tensors="pt",
    )


class BGEGemmaReranker(pt.Transformer):
    def __init__(self, corpus_df: pd.DataFrame, model_name: str, batch_size: int = 4, max_length: int = 512):
        super().__init__()
        cdf = corpus_df[["docno", "text"]].copy()
        cdf["docno"] = cdf["docno"].astype(str)
        cdf["text"] = cdf["text"].astype(str)
        self.docno2text = dict(zip(cdf["docno"], cdf["text"]))
        self.batch_size = int(batch_size)
        self.max_length = int(max_length)
        if torch.cuda.is_available():
            self.device = "cuda"
        elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
            self.device = "mps"
        else:
            self.device = "cpu"
        model_kwargs = {"low_cpu_mem_usage": True}
        if self.device in {"cuda", "mps"}:
            model_kwargs["dtype"] = torch.float16
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(model_name, **model_kwargs).to(self.device)
        self.model.eval()
        self.yes_loc = self.tokenizer("Yes", add_special_tokens=False)["input_ids"][0]
        print(f"Loaded {model_name} on {self.device}")

    def _score_pairs(self, pairs):
        scores = []
        for start in range(0, len(pairs), self.batch_size):
            batch_pairs = pairs[start:start + self.batch_size]
            inputs = build_llm_inputs(batch_pairs, self.tokenizer, max_length=self.max_length)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            with torch.no_grad():
                logits = self.model(**inputs, return_dict=True).logits[:, -1, self.yes_loc]
            batch_scores = logits.float().detach().cpu().numpy().astype(float).tolist()
            scores.extend(batch_scores)
            del inputs, logits
            if self.device == "cuda":
                torch.cuda.empty_cache()
            elif self.device == "mps":
                torch.mps.empty_cache()
        return np.asarray(scores, dtype=float)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = ensure_run(df)
        if "query" not in df.columns:
            raise ValueError("BGEGemmaReranker needs query column, merge topics first")
        out_groups = []
        for qid, g in df.groupby("qid", sort=False):
            qtext = str(g["query"].iloc[0])
            docnos = g["docno"].astype(str).tolist()
            doc_texts = [self.docno2text.get(docno, "") for docno in docnos]
            pairs = [(qtext, doc_text) for doc_text in doc_texts]
            scores = self._score_pairs(pairs)
            gg = g.copy()
            gg["score"] = scores
            gg = gg.sort_values("score", ascending=False)
            gg["rank"] = np.arange(1, len(gg) + 1)
            out_groups.append(gg)
        return pd.concat(out_groups, ignore_index=True)


def eval_at_k(pipelines: List[Tuple[str, pt.Transformer]], topics_df: pd.DataFrame, qrels_df: pd.DataFrame, k_eval: int) -> pd.DataFrame:
    metrics = [
        f"ndcg_cut.{k_eval}",
        "recip_rank",
        f"P.{k_eval}",
        f"recall.{k_eval}",
        f"map_cut.{k_eval}",
    ]
    df = pt.Experiment(
        [p for _, p in pipelines],
        topics_df,
        qrels_df,
        eval_metrics=metrics,
        names=[n for n, _ in pipelines],
        verbose=True,
        validate="ignore",
    )
    df = df.rename(columns={
        f"ndcg_cut.{k_eval}": f"nDCG@{k_eval}",
        "recip_rank": f"MRR@{k_eval}",
        f"P.{k_eval}": f"Prec@{k_eval}",
        f"recall.{k_eval}": f"Recall@{k_eval}",
        f"map_cut.{k_eval}": f"MAP@{k_eval}",
    }).copy()
    p = df[f"Prec@{k_eval}"].astype(float)
    r = df[f"Recall@{k_eval}"].astype(float)
    df[f"F1@{k_eval}"] = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)
    cols = ["name", f"nDCG@{k_eval}", f"MRR@{k_eval}", f"Prec@{k_eval}", f"Recall@{k_eval}", f"F1@{k_eval}", f"MAP@{k_eval}"]
    return df[cols]


def to_table1_style(metrics_df: pd.DataFrame, ann_name: str, k_eval: int = 50) -> pd.DataFrame:
    df = metrics_df.copy()
    ann_ndcg = float(df.loc[df["name"] == ann_name, f"nDCG@{k_eval}"].iloc[0])
    df["Delta"] = df[f"nDCG@{k_eval}"].astype(float) - ann_ndcg
    df["Pct"] = np.where(ann_ndcg != 0, 100.0 * df["Delta"] / ann_ndcg, 0.0)
    out = pd.DataFrame({
        "Method": df["name"],
        "nDCG": df[f"nDCG@{k_eval}"].astype(float),
        "Delta": df["Delta"].astype(float),
        "%": df["Pct"].astype(float),
        "MRR": df[f"MRR@{k_eval}"].astype(float),
        "P": df[f"Prec@{k_eval}"].astype(float),
        "R": df[f"Recall@{k_eval}"].astype(float),
        "F1": df[f"F1@{k_eval}"].astype(float),
        "MAP": df[f"MAP@{k_eval}"].astype(float),
    })
    return out


qb = pd.read_csv(QB_PATH).fillna("N/A")
queries = pd.read_csv(QUERIES_PATH).fillna("")
qrels = pd.read_csv(QRELS_PATH, sep="\t").fillna(0)

qb["docno"] = qb["test_item_id"].astype(str)
qb["raw_text"] = qb.apply(build_doc_text, axis=1)
qb["text"] = qb["raw_text"].map(normalize_text)
corpus = qb[["docno", "text"]].copy()

topics = queries.rename(columns={"queries": "query"})[["qid", "query"]].copy()
topics["qid"] = topics["qid"].astype(str)
topics["query"] = topics["query"].astype(str).map(normalize_text)

qrels = qrels.rename(columns={"rel": "label"})[["qid", "docno", "label"]].copy()
qrels["qid"] = qrels["qid"].astype(str)
qrels["docno"] = qrels["docno"].astype(str)
qrels["label"] = qrels["label"].astype(int)

dense100 = FaissDenseRetriever(corpus, model_name=DENSE_MODEL, topk=DENSE_TOPK)
bge_gemma = BGEGemmaReranker(corpus, model_name=RERANKER_MODEL, batch_size=RERANK_BATCH, max_length=MAX_LENGTH)
cut50 = cut_k(FINAL_TOPK)
add_query = add_query_col(topics)

pipelines: List[Tuple[str, pt.Transformer]] = [
    ("ANN@100->@50", dense100 >> cut50),
    ("dense@100>>BGEGemma(rescore)->@50", dense100 >> add_query >> bge_gemma >> cut50),
]

metrics_df = eval_at_k(pipelines, topics, qrels, k_eval=FINAL_TOPK)
metrics_df = metrics_df.sort_values(f"nDCG@{FINAL_TOPK}", ascending=False).reset_index(drop=True)
table_df = to_table1_style(metrics_df, ann_name="ANN@100->@50", k_eval=FINAL_TOPK)
table_df = table_df.sort_values("nDCG", ascending=False).reset_index(drop=True)
new_row_df = table_df[table_df["Method"] == "dense@100>>BGEGemma(rescore)->@50"].reset_index(drop=True)

metrics_path = ART_DIR / "bge_gemma_reranker_metrics_raw.csv"
compare_path = ART_DIR / "bge_gemma_reranker_table1_compare.csv"
row_path = ART_DIR / "bge_gemma_reranker_table1_row.csv"

metrics_df.to_csv(metrics_path, index=False)
table_df.to_csv(compare_path, index=False)
new_row_df.to_csv(row_path, index=False)

print(metrics_df.to_string(index=False))
print()
print(new_row_df.to_string(index=False))
print()
print("Saved:", metrics_path)
print("Saved:", compare_path)
print("Saved:", row_path)

/Users/user/miniconda3/envs/KG-re-ranking/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]


Batches:   0%|          | 0/88 [00:00<?, ?it/s]

Batches:   1%|          | 1/88 [00:05<07:23,  5.09s/it]

Batches:   2%|▏         | 2/88 [00:09<06:58,  4.86s/it]

...

/Users/user/miniconda3/envs/KG-re-ranking/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:2752: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


/Users/user/miniconda3/envs/KG-re-ranking/lib/python3.11/site-packages/transformers/tokenization_utils_base.py:2752: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


pt.Experiment: 100%|██████████| 2/2 [03:54<00:00, 137.62s/system]

pt.Experiment: 100%|██████████| 2/2 [03:54<00:00, 117.16s/system]

                             name  nDCG@50   MRR@50  Prec@50  Recall@50    F1@50   MAP@50
dense@100>>BGEGemma(rescore)->@50 0.600791 0.911111 0.518667   0.452146 0.483127 0.366744
                     ANN@100->@50 0.500596 0.794444 0.462667   0.411309 0.435479 0.273427

                           Method     nDCG    Delta         %      MRR        P        R       F1      MAP
dense@100>>BGEGemma(rescore)->@50 0.600791 0.100195 20.015186 0.911111 0.518667 0.452146 0.483127 0.366744

Saved: /Users/user/Submissions/BEA-2026/artifacts/bge_gemma_reranker/bge_gemma_reranker_metrics_raw.csv
Saved: /Users/user/Submissions/BEA-2026/artifacts/bge_gemma_reranker/bge_gemma_reranker_table1_compare.csv
Saved: /Users/user/Submissions/BEA-2026/artifacts/bge_gemma_reranker/bge_gemma_reranker_table1_row.csv
